In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
import pandas as pd
import glob
import os
import numpy as np

In [2]:
df = pd.read_csv(r"H:\GitHub\AD_Behavioral_Modeling\data\I-80_VehTrajectoryData\0400pm-0415pm\trajectories-0400-0415.txt", 
                  sep=r"\s+", header=None, names=["Vehicle_ID", "Frame_ID", "Total_Frames", "Global_Time", "Local_X", "Local_Y", "Global_X", 
                                "Global_Y", "v_len", "v_width", "v_class", "v_vel", "v_acc", "Lane_ID", "Preceeding", 
                                "Following", "Space_Headway", "Time_Headway"])
df['Vehicle_Global_ID'] = df.Vehicle_ID
df['Frame_Global_ID'] = df.Frame_ID

In [27]:
# 1. Sort for consistent time-series calculations
df = df.sort_values(['Vehicle_Global_ID', 'Frame_Global_ID']).reset_index(drop=True)
dt = 0.1  # 100ms interval

In [31]:
df[['v_vel', 'a_long', 'v_lat']][0:5]

,v_vel,a_long,v_lat
0,12.5,NaN,NaN
1,12.5,0.0,0.54
2,12.5,0.0,0.53
3,12.5,0.0,0.54
4,12.5,0.0,0.53


In [28]:
# Lateral Velocity: Change in Local_X over time
df['v_lat'] = df.groupby('Vehicle_Global_ID')['Local_X'].diff() / dt

In [29]:
# Using v_vel (longitudinal velocity)
df['a_long'] = df.groupby('Vehicle_Global_ID')['v_vel'].diff() / dt

In [30]:
# Create a mapping for quick lookup of lead vehicle stats per frame
# We need v_vel and Local_Y of the 'Preceeding' vehicle ID
lead_info = df[['Frame_Global_ID', 'Vehicle_Global_ID', 'v_vel', 'Local_Y', 'v_len']].copy()
lead_info = lead_info.rename(columns={
        'Vehicle_Global_ID': 'Preceeding',
        'v_vel': 'v_vel_lead',
        'Local_Y': 'Local_Y_lead',
        'v_len': 'v_len_lead'
    })

In [32]:
lead_info.columns

Index(['Frame_Global_ID', 'Preceeding', 'v_vel_lead', 'Local_Y_lead',
       'v_len_lead'],
      dtype='object')

In [33]:
# Merge lead vehicle info onto the main dataframe
df = pd.merge(df, lead_info, on=['Frame_Global_ID', 'Preceeding'], how='left')

In [4]:
df[['Frame_Global_ID', 'Preceeding', 'Vehicle_Global_ID',  'v_vel','v_lat',
       'a_long','rel_speed']][df.Vehicle_Global_ID == 11]

,Frame_Global_ID,Preceeding,Vehicle_Global_ID,v_vel,v_lat,a_long,rel_speed
3461,57,1,11,3.819144,0.000000,0.00000,0.140208
3462,58,1,11,3.819144,-0.295656,0.00000,0.137160
3463,59,1,11,3.819144,-0.295656,0.00000,0.042672
3464,60,1,11,3.819144,-0.298704,0.00000,-0.222504
3465,61,1,11,3.819144,-0.301752,0.00000,-0.676656
...,...,...,...,...,...,...,...
4320,916,0,11,9.521952,0.067056,1.15824,0.000000
4321,917,0,11,9.512808,0.067056,-0.09144,0.000000
4322,918,0,11,9.512808,0.064008,0.00000,0.000000
4323,919,0,11,9.512808,0.064008,0.00000,0.000000


In [6]:
df.columns

Index(['Vehicle_ID', 'Frame_ID', 'Total_Frames', 'Global_Time', 'Local_X',
       'Local_Y', 'Global_X', 'Global_Y', 'v_len', 'v_width', 'v_class',
       'v_vel', 'v_acc', 'Lane_ID', 'Preceeding', 'Following', 'Space_Headway',
       'Time_Headway', 'Vehicle_Global_ID', 'Frame_Global_ID', 'v_lat',
       'a_long', 'rel_speed', 'actual_gap', 'TTC'],
      dtype='object')

In [36]:
 df['rel_speed'] = df['v_vel_lead'] - df['v_vel']

In [38]:
    # 2. Space Headway (Actual Gap)
    # NGSIM 'Space_Headway' is center-to-center. 
    # Gap = (Y_lead - Y_ego) - (Len_lead/2 + Len_ego/2)
    df['actual_gap'] = (df['Local_Y_lead'] - df['Local_Y']) - (df['v_len_lead']/2 + df['v_len']/2)
    # Fill cases where there is no lead vehicle with a large constant
    df['actual_gap'] = df['actual_gap'].fillna(1000) 

In [39]:
    # 3. Time-to-Collision (TTC)
    # Formula: Gap / Relative_Velocity (only if Relative_Velocity is negative, i.e., closing)
    # Relative Velocity here is (v_ego - v_lead)
    closing_vel = df['v_vel'] - df['v_vel_lead']
    df['TTC'] = np.where(
        (closing_vel > 0) & (df['actual_gap'] > 0),
        df['actual_gap'] / closing_vel,
        100 # Default value for no collision risk
    )
    
    # Cap TTC at 100 seconds to avoid infinities
    df['TTC'] = df['TTC'].clip(upper=100)

    # Clean up helper columns
    df = df.drop(columns=['v_vel_lead', 'Local_Y_lead', 'v_len_lead'])

In [40]:
df.shape

(1262678, 25)

In [43]:
    df['rel_speed'] = df['rel_speed'].fillna(0) 
    df['v_lat'] = df['v_lat'].fillna(0) 
    df['a_long'] = df['a_long'].fillna(0) 

In [48]:
cols_to_scale = ['Local_X',
 'Local_Y',
 'Global_X',
 'Global_Y',
 'v_len',
 'v_width',
 'Space_Headway',
 'actual_gap',
 'v_vel',
 'v_lat',
 'rel_speed',
 'v_acc',
 'a_long']


In [50]:
    # Apply conversion only to columns that exist in the dataframe
    # Define conversion factor (feet to meters)
    FT_TO_M = 0.3048
    existing_cols = [c for c in cols_to_scale if c in df.columns]
    df[existing_cols] = df[existing_cols] * FT_TO_M

In [4]:
import os
os.chdir(os.path.expanduser("H:\GitHub\AD_Behavioral_Modeling"))

<>:2: SyntaxWarning: invalid escape sequence '\G'
<>:2: SyntaxWarning: invalid escape sequence '\G'
C:\Users\ranji\AppData\Local\Temp\ipykernel_10952\2760069814.py:2: SyntaxWarning: invalid escape sequence '\G'
  os.chdir(os.path.expanduser("H:\GitHub\AD_Behavioral_Modeling"))


In [3]:
from extract_features import calculate_features
df = calculate_features(df)

Calculating Individual Dynamics...
Calculating Lead Vehicle Features...
